# 10 - LlamaIndex Governed Retrieval Pattern

A retrieval function that is governance-aware end to end: the planner maps a
question to SQL + a canonical scenario, Metatate validates it, and only a
`pass`/revised query reaches the retriever. Wrap `governed_retrieval` as a
LlamaIndex `FunctionTool` and the framework routes through the same gate
(`framework_runtime/llamaindex_acceptance.py` proves it).


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None, schema="public"):
    ref = {"database": "acmecloud_demo", "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
SAFE_SQL = "SELECT region, SUM(arr) FROM customers GROUP BY region"

def plan_retrieval(question):
    q = question.lower()
    if "marketing" in q:
        return (
            "SELECT customer_name, email FROM customers WHERE marketing_consent = 'opted_in'",
            "purpose.prohibited_use",
        )
    if "email" in q:
        return (
            "SELECT customer_name, email FROM customers WHERE region = 'EU'",
            "purpose.allowed_use",
        )
    return (SAFE_SQL, "purpose.allowed_use")

def governed_retrieval(question):
    sql, scenario_key = plan_retrieval(question)
    answer = client.validate_query_context(
        sql,
        scenario_key=scenario_key,
        default_database="acmecloud_demo",
        default_schema="public",
    )
    if answer["verdict"] == "fail":
        return {"question": question, "retrieved": None, "verdict": "fail"}
    final_sql = sql if answer["verdict"] == "pass" else SAFE_SQL
    return {"question": question, "retrieved": final_sql, "verdict": answer["verdict"]}


In [ ]:
for question in [
    "What is ARR by region?",
    "Show EU customers and their email addresses.",
    "Pull the marketing outreach list.",
]:
    print(governed_retrieval(question))
